In [ ]:
library(dplyr, warn.conflicts = FALSE)
library(tidyr, warn.conflicts = FALSE)
library(purrr, warn.conflicts = FALSE)
library(tibble, warn.conflicts = FALSE)
#library(furrr)
library(future)
library(readr)
library(edgeR)
library(limma)
library(dplyr)
library(statmod)
library(data.table)
library(arrow)

In [ ]:
# Function to convert drug names to valid R variable names
make_valid_names <- function(names_vector) {
  # Replace invalid characters with underscores
  valid_names <- gsub("[^[:alnum:]_]", "_", names_vector)
  
  # Ensure the names start with a letter (required for R variable names)
  valid_names <- make.names(valid_names)
  
  return(valid_names)
}

In [ ]:
# Function to clean the column names
clean_column_names <- function(names_vector) {
  # Remove "factor(valid_drug_names)X" prefix
  cleaned_names <- gsub("^factor\\(valid_drug_names\\)X", "", names_vector)
  
  # Replace multiple underscores with a single underscore
  cleaned_names <- gsub("_+", "_", cleaned_names)
  
  return(cleaned_names)
}

In [ ]:
# Function to clean the column names
clean_contrast_names <- function(names_vector) {
  # Remove leading underscores (or other unwanted characters) and clean the names
  cleaned_names <- gsub("^_", "", names_vector)  # Remove leading underscores
  cleaned_names <- make.names(cleaned_names)  # Ensure valid R names
  
  # Optionally, remove additional unwanted characters if needed
  cleaned_names <- gsub("[^[:alnum:]_]", "_", cleaned_names)  # Replace non-alphanumeric characters with underscores
  
  return(cleaned_names)
}

In [ ]:
# Set the path to your folder
folder_path <- "/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/sciplex2"

# Get all files in the folder
files <- list.files(path = folder_path, full.names = TRUE)
sample_names <- sub(".*/(.*)_.*\\.csv", "\\1", files)
celllines = sort(unique(unname(sample_names)))
celllines

In [ ]:
Sys.sleep(1000)

In [ ]:
for (cellline in celllines){
    tryCatch({
    file0 = '/lustre/groups/ml01/workspace/manuel.gander/data/prelimma/sciplex2/'
    path = paste0(file0, cellline)

    X <- as.matrix(fread(paste0(path, "_X.csv"), header = FALSE, skip = 1))
    var <- read_csv(paste0(path, "_var.csv"))
    obs <- read_csv(paste0(path, "_obs.csv"))

    valid_drug_names <- make_valid_names(obs$drugname_drugconc)
    plate <- factor(obs$plate)

    # Create the design matrix based on the cell types
    design_matrix <- model.matrix(~ 0 + factor(valid_drug_names) + plate)
    colnames(design_matrix) <- clean_column_names(colnames(design_matrix))
    colnames(design_matrix) <- clean_contrast_names(colnames(design_matrix))


    # Create DGEList object
    d0 <- DGEList(counts = t(X))
    keep_genes <- filterByExpr(d0, design = design_matrix)
    d0 <- d0[keep_genes, , keep.lib.sizes = FALSE]
    d0 <- calcNormFactors(d0)

    # Perform the Voom transformation and fit the model
    v <- voom(d0, design = design_matrix, plot = FALSE)
    fit <- lmFit(v, design_matrix)

    # Output the results
    summary(fit)

    # get proper DMSO name
    conditions <- setdiff(unique(colnames(design_matrix)), "factor_valid_drug_names_control_0_0")

    # Initialize a list to store results
    de_results_list <- list()

    # Loop through conditions
    for (condition in conditions) {
        cat("Doing condition", condition, "\n")
        flush.console()  # Force the output to be printed immediately
        # Create contrast formula
        contrast_formula <- paste0(condition, " - ", "factor_valid_drug_names_control_0_0")
        contr <- makeContrasts(contrasts = contrast_formula, levels = colnames(design_matrix))

        # Fit the contrast and apply eBayes
        contrast_fit <- contrasts.fit(fit, contr)
        contrast_fit <- eBayes(contrast_fit, robust = TRUE)

        # Get the top table of results
        top_table <- topTable(contrast_fit, number = Inf, sort.by = "none", adjust.method = "BH")
        keep_genes[1] <- FALSE
        rownames(top_table) <- var$'ensembl_id...1'[keep_genes]

        # Add additional metadata
        top_table$gene <- rownames(top_table)
        top_table$condition <- condition
        top_table$control_type <- "factor_valid_drug_names_control_0_0"
        top_table$cellline <- cellline

        # Store in list (only if significant)
        if(nrow(top_table) > 0) {
            de_results_list[[condition]] <- top_table
        }
    }

    de_results_combined <- do.call(rbind, de_results_list)

    output_file <- paste0("/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/sciplex2/", 
        cellline, "_conditions.parquet")
    write_parquet(tibble(condition = conditions), output_file)


    output_file <- paste0("/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/sciplex2/", 
        cellline, "_differential_expression_results.parquet")
    write_parquet(de_results_combined, output_file)
    
    })
}